<a href="https://colab.research.google.com/github/DhimanTarafdar/gallbladder_cancer_research/blob/main/Gallbladder_Cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install** and load required packages

In [1]:
!pip install rpy2 --quiet

import pandas as pd
import numpy as np
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, numpy2ri, r
from rpy2.robjects.packages import importr

# Install R packages (only needs to run once per session)
robjects.r('''
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos="http://cran.r-project.org")
BiocManager::install(c("limma", "edgeR"), update = FALSE, ask = FALSE)
''')

limma = importr('limma')
pandas2ri.activate()
numpy2ri.activate()
print("Setup done ✅")

(as ‘lib’ is unspecified)







	‘/tmp/Rtmpte2y4p/downloaded_packages’

'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com











	‘/tmp/Rtmpte2y4p/downloaded_packages’



Setup done ✅


# **Load GSE139682 (normalized RPKM data)**

In [2]:
df_139682 = pd.read_csv('GSE139682_all.rpkm.txt', sep='\t')

# Get normal and tumor sample columns automatically
normal_cols_139682 = [c for c in df_139682.columns if c.startswith('Normal')]
tumor_cols_139682  = [c for c in df_139682.columns if c.startswith('Tumor')]

print(f"GSE139682 -> Normal: {len(normal_cols_139682)}, Tumor: {len(tumor_cols_139682)}")

GSE139682 -> Normal: 10, Tumor: 10


# **Load GSE202479 (raw count data) + build labels**

In [5]:
def extract_full_metadata_line(series_matrix_path, keyword):
    # This function reads one metadata line from the series_matrix file
    with open(series_matrix_path, 'r', errors='ignore') as f:
        for line in f:
            if line.startswith(keyword):
                parts = line.strip().split('\t')
                parts = [p.strip('"') for p in parts]
                return parts[1:]  # skip the label, keep only values
    return None

import re

titles = extract_full_metadata_line('GSE202479_series_matrix.txt', '!Sample_title')
tissue_type = extract_full_metadata_line('GSE202479_series_matrix.txt', '!Sample_characteristics_ch1')

# Map sample code (like N8, T1) to normal/tumor/exclude
label_map_202479 = {}
for title, tissue in zip(titles, tissue_type):
    code = re.search(r'\[(.*?)\]', title).group(1)
    tissue_clean = tissue.replace('tissue type: ', '').strip()
    if tissue_clean == 'normal gallbladder':
        label_map_202479[code] = 'normal'
    elif tissue_clean == 'advanced gallbladder cancer':
        label_map_202479[code] = 'tumor'
    else:
        label_map_202479[code] = 'exclude'   # skip inflammation, adenoma, early stage

# Build final column lists
normal_codes = [k for k, v in label_map_202479.items() if v == 'normal']
tumor_codes  = [k for k, v in label_map_202479.items() if v == 'tumor']

normal_cols_202479 = [f"{c}_count" for c in normal_codes]
tumor_cols_202479  = [f"{c}_count" for c in tumor_codes]

df_202479 = pd.read_csv('GSE202479_gene_expression_anno.txt', sep='\t')

print(f"GSE202479 -> Normal: {len(normal_cols_202479)}, Tumor: {len(tumor_cols_202479)}")

GSE202479 -> Normal: 3, Tumor: 5


In [6]:
selected_cols = ['Gene', 'Symbol'] + normal_cols_202479 + tumor_cols_202479

print(df_202479[normal_cols_202479 + tumor_cols_202479].head())

   N8_count  N10_count  N20_count  T1_count  T19_count  T22_count  T27_count  \
0         0          0          0         0          0          0          0   
1        86         56        175        25         30         42         38   
2      1176        975       1705      1151       1253       1631       1334   
3      1717       1582       2262      2152       1397       1729       3035   
4         3          2          5         1          0          4          7   

   T32_count  
0          0  
1         84  
2       1238  
3       4213  
4          0  


In [8]:
print("Dataset Shape:", df_202479.shape)


Dataset Shape: (50868, 50)


# **DEG analysis functions**

In [10]:
def run_limma_normalized(expr_df, gene_col, normal_cols, tumor_cols, dataset_name="Dataset"):
    # Use this for already-normalized data (like RPKM/FPKM)
    expr_matrix = expr_df[normal_cols + tumor_cols].values
    expr_matrix = np.log2(expr_matrix + 1)  # log transform, avoid log(0)

    genes = expr_df[gene_col].values
    n_normal, n_tumor = len(normal_cols), len(tumor_cols)

    r.assign('expr_mat', expr_matrix)
    r.assign('gene_ids', robjects.StrVector(genes.astype(str)))

    r(f'''
    rownames(expr_mat) <- gene_ids
    group <- factor(c(rep("normal", {n_normal}), rep("tumor", {n_tumor})), levels=c("normal","tumor"))
    design <- model.matrix(~group)
    fit <- lmFit(expr_mat, design)
    fit <- eBayes(fit)
    result <- topTable(fit, coef=2, number=Inf, sort.by="P")
    ''')

    result_r = r('result')
    with (robjects.default_converter + pandas2ri.converter).context():
        result_df = robjects.conversion.get_conversion().rpy2py(result_r)

    result_df.index.name = 'GeneID'
    result_df = result_df.reset_index()
    print(f"{dataset_name}: {result_df.shape[0]} genes tested")
    return result_df


def run_limma_voom(count_df, gene_col, normal_cols, tumor_cols, dataset_name="Dataset"):
    # Use this for raw count data
    count_matrix = count_df[normal_cols + tumor_cols].values
    genes = count_df[gene_col].values
    n_normal, n_tumor = len(normal_cols), len(tumor_cols)

    r.assign('count_mat', count_matrix)
    r.assign('gene_ids', robjects.StrVector(genes.astype(str)))

    r(f'''
    rownames(count_mat) <- gene_ids
    group <- factor(c(rep("normal", {n_normal}), rep("tumor", {n_tumor})), levels=c("normal","tumor"))
    design <- model.matrix(~group)

    library(edgeR)
    dge <- DGEList(counts=count_mat)
    keep <- filterByExpr(dge, design)
    dge <- dge[keep,, keep.lib.sizes=FALSE]
    dge <- calcNormFactors(dge)

    v <- voom(dge, design)
    fit <- lmFit(v, design)
    fit <- eBayes(fit)
    result <- topTable(fit, coef=2, number=Inf, sort.by="P")
    ''')

    result_r = r('result')
    with (robjects.default_converter + pandas2ri.converter).context():
        result_df = robjects.conversion.get_conversion().rpy2py(result_r)

    result_df.index.name = 'GeneID'
    result_df = result_df.reset_index()
    print(f"{dataset_name}: {result_df.shape[0]} genes tested (after low-expression filter)")
    return result_df

# **Run DEG analysis on both datasets**

In [11]:
deg_139682 = run_limma_normalized(df_139682, 'GeneID', normal_cols_139682, tumor_cols_139682, "GSE139682")
deg_202479 = run_limma_voom(df_202479, 'gene_id', normal_cols_202479, tumor_cols_202479, "GSE202479")

GSE139682: 41762 genes tested


GSE202479: 21273 genes tested (after low-expression filter)


# **Filter significant DEGs**

In [12]:
def filter_significant_degs(deg_df, logfc_col='logFC', padj_col='adj.P.Val',
                              logfc_thresh=1, padj_thresh=0.05):
    # Keep only genes that are statistically significant AND have a strong fold change
    sig = deg_df[(deg_df[padj_col] < padj_thresh) & (deg_df[logfc_col].abs() > logfc_thresh)]
    print(f"Significant DEGs: {sig.shape[0]} out of {deg_df.shape[0]}")
    return sig

sig_139682 = filter_significant_degs(deg_139682)
sig_202479 = filter_significant_degs(deg_202479)

Significant DEGs: 650 out of 41762
Significant DEGs: 1033 out of 21273


# **Map all gene IDs to Symbol (common gene naming system)**

In [14]:
df_annot = pd.read_csv('Human.GRCh38.p13.annot.tsv', sep='\t', low_memory=False)
ensembl_to_symbol = dict(zip(df_annot['EnsemblGeneID'], df_annot['Symbol']))

# GSE139682 already uses Symbol, no conversion needed
sig_139682_genes = set(sig_139682['GeneID'])

# GSE202479 uses Ensembl ID, convert to Symbol
sig_202479_genes = set(sig_202479['GeneID'].map(ensembl_to_symbol).dropna())

print(f"GSE139682 mapped genes: {len(sig_139682_genes)}")
print(f"GSE202479 mapped genes: {len(sig_202479_genes)}")

GSE139682 mapped genes: 650
GSE202479 mapped genes: 879
